# Notebook 03 — Preprocessing, Rate Normalization, Feature Engineering & Leakage Audit

**Chuẩn hóa tiền xử lý thống nhất cho cả Weekly (`302W`) & Daily (`2,106D`):**
1. **Chống rò rỉ tuyệt đối:** Loại bỏ `is_holiday_zero(t)` khỏi đặc trưng (chỉ giữ `is_rs_holiday` biết trước & `is_holiday_zero_lag1`); bỏ `uv_index_max` (100% NULL) và dùng thuần **`.ffill()`** (không `.bfill()`).
2. **Calendar-Exposure Rate Normalization:** Tính `qty_rate = qty / active_days` và sinh toàn bộ `F1` lags/rolling trên `qty_rate` (`.shift(1)`), giúp tuần khuyết lịch `W1` (`4` ngày) và `W302` (`2` ngày) không làm méo động lực học mô hình. `active_days` chỉ đóng vai trò hệ số nhân xác định ở bước phục hồi `ŷ = r̂ · active_days`.
3. **Compact Physics-Orthogonal ERA5 (`f3_compact_cols`, 7 biến):** Gọn, giảm cộng tuyến từ `19 → 7` biến vật lý cốt lõi (`temp_mean, hdd, humidity_mean, daylight_duration_s, temp_shock_2w, temp_mean_lag1, humidity_mean_lag1`), đồng thời giữ đủ `f3_lagged_only_cols` (`8` biến) & `f3_all_cols` (`19` biến) trong `feature_catalog.json`.
4. **Automated Perturbation Leakage Unit-Test (`max_abs_diff == 0.0`)** trên cả Weekly & Daily.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")

from pathlib import Path
import os

def resolve_art_dir() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "BTL_BI").is_dir():
            d = p / "BTL_BI" / "artifacts" / "forecasting"
            d.mkdir(parents=True, exist_ok=True)
            return d
        if p.name == "BTL_BI":
            d = p / "artifacts" / "forecasting"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = cwd / "artifacts" / "forecasting"
    d.mkdir(parents=True, exist_ok=True)
    return d

ART_DIR = resolve_art_dir()
print("Artifacts dir:", ART_DIR)

df_sales = pd.read_parquet(ART_DIR / "raw_sales_5631.parquet")
df_weather = pd.read_parquet(ART_DIR / "raw_weather_5631.parquet")
df_hol = pd.read_parquet(ART_DIR / "raw_holidays_rs.parquet")

df_sales["local_date"] = pd.to_datetime(df_sales["local_date"])
df_weather["local_date"] = pd.to_datetime(df_weather["local_date"])
df_hol["local_date"] = pd.to_datetime(df_hol["local_date"])

In [ ]:
# 1. Outlier Audit (IQR & |z| > 3s) trên Weekly W-SUN để chứng minh giữ nguyên OBSERVED
w_check = (
    df_sales.set_index("local_date").groupby("atc_code")["qty"]
    .resample("W-SUN").sum().reset_index()
)
out_rows = []
for atc, g in w_check.groupby("atc_code"):
    y = g["qty"]
    q1, q3 = y.quantile(0.25), y.quantile(0.75)
    iqr = q3 - q1
    n_iqr = int(((y < q1 - 1.5 * iqr) | (y > q3 + 1.5 * iqr)).sum())
    z = (y - y.mean()) / y.std(ddof=1)
    n_3s = int((z.abs() > 3.0).sum())
    peak_date = str(g.loc[y.idxmax(), "local_date"].date())
    out_rows.append({"atc_code": atc, "weeks": len(g), "iqr_outliers": n_iqr, "z_gt_3s": n_3s, "max_qty": round(float(y.max()), 1), "max_week": peak_date})
print("Outlier Audit (Seasonal peaks preserved as OBSERVED; no Grubbs 3s clipping):")
display(pd.DataFrame(out_rows))

In [ ]:
# 2. Lưới Daily liên tục (2,106D x 8 ATC) + thuần .ffill() thời tiết (ZERO .bfill()) + Resample W-SUN (302W)
w_cols = [
    "temp_mean", "temp_min", "temp_max", "apparent_temp_mean",
    "humidity_mean", "precip_sum", "wind_speed_max", "wind_speed_mean",
    "cloud_cover_mean", "pressure_msl_mean",
    "shortwave_radiation_sum", "daylight_duration_s",
]

daily_tot = df_sales.groupby("local_date")["qty"].transform("sum")
df_daily_base = (
    df_sales.assign(is_holiday_zero_obs=(daily_tot == 0).astype(int), active_days=1)
    .merge(df_weather[["local_date"] + w_cols], on="local_date", how="left")
    .merge(df_hol.assign(is_rs_holiday=1)[["local_date", "is_rs_holiday"]], on="local_date", how="left")
    .assign(is_rs_holiday=lambda d: d["is_rs_holiday"].fillna(0).astype(int))
    .sort_values(["atc_code", "local_date"])
    .reset_index(drop=True)
)
df_daily_base["day_idx"] = df_daily_base.groupby("atc_code").cumcount()

df_daily_base[w_cols] = df_daily_base.groupby("atc_code")[w_cols].ffill()
assert df_daily_base[w_cols].isna().sum().sum() == 0, "Unexpected NaN in weather after .ffill()!"

agg_rules = {
    "qty": "sum",
    "is_holiday_zero_obs": "sum",
    "is_rs_holiday": "sum",
    "temp_mean": "mean",
    "temp_min": "min",
    "temp_max": "max",
    "humidity_mean": "mean",
    "precip_sum": "sum",
    "pressure_msl_mean": "mean",
    "daylight_duration_s": "mean",
    "active_days": "sum",
}
df_weekly = (
    df_daily_base.set_index("local_date")
    .groupby("atc_code")
    .resample("W-SUN")
    .agg(agg_rules)
    .reset_index()
    .sort_values(["atc_code", "local_date"])
    .reset_index(drop=True)
)
df_weekly["week_idx"] = df_weekly.groupby("atc_code").cumcount()
assert (df_weekly.groupby("atc_code")["week_idx"].count() == 302).all()
assert (df_daily_base.groupby("atc_code")["day_idx"].count() == 2106).all()
assert (df_weekly["active_days"] > 0).all()
print(f"Base daily: {df_daily_base.shape}, Base weekly: {df_weekly.shape}, active_days dist: {df_weekly['active_days'].value_counts().to_dict()}")

In [ ]:
# 3. Hàm dựng Feature Weekly (Rate-Normalized F1 + F2 + F3 Compact/LagOnly/All) & Daily

def build_weekly_features(df_in: pd.DataFrame):
    df = df_in.sort_values(["atc_code", "local_date"]).copy()
    df["qty_rate"] = df["qty"] / df["active_days"].astype(float)
    tot_r = df.groupby("local_date")["qty_rate"].transform("sum")
    df["atc_share_raw"] = df["qty_rate"] / tot_r.replace(0, np.nan)

    # F1 — Lịch sử nội sinh tính trên tốc độ ngày qty_rate (.shift(1) trước khi rolling/ewm)
    f1_cols = []
    for k in [1, 2, 3, 4, 8, 12, 52]:
        col = f"lag_{k}"
        df[col] = df.groupby("atc_code")["qty_rate"].shift(k)
        f1_cols.append(col)

    s1 = df.groupby("atc_code")["qty_rate"].shift(1)
    df["rolling_mean_4"] = s1.groupby(df["atc_code"]).transform(lambda s: s.rolling(4, min_periods=4).mean())
    df["rolling_std_4"] = s1.groupby(df["atc_code"]).transform(lambda s: s.rolling(4, min_periods=4).std())
    df["rolling_max_4"] = s1.groupby(df["atc_code"]).transform(lambda s: s.rolling(4, min_periods=4).max())
    df["rolling_mean_12"] = s1.groupby(df["atc_code"]).transform(lambda s: s.rolling(12, min_periods=12).mean())
    df["rolling_std_12"] = s1.groupby(df["atc_code"]).transform(lambda s: s.rolling(12, min_periods=12).std())
    df["ewma_4"] = s1.groupby(df["atc_code"]).transform(lambda s: s.ewm(span=4, adjust=False, min_periods=1).mean())
    df["zero_ratio_12"] = (s1 == 0).astype(float).groupby(df["atc_code"]).transform(lambda s: s.rolling(12, min_periods=12).mean())
    f1_cols += ["rolling_mean_4", "rolling_std_4", "rolling_max_4", "rolling_mean_12", "rolling_std_12", "ewma_4", "zero_ratio_12"]

    # F2 — Lịch biết trước & Tỷ trọng quá khứ (KHÔNG dùng is_holiday_zero(t), KHÔNG đưa active_days làm feature cây)
    woy = df["local_date"].dt.isocalendar().week.astype(int)
    df["week_sin52"] = np.sin(2.0 * np.pi * woy / 52.0)
    df["week_cos52"] = np.cos(2.0 * np.pi * woy / 52.0)
    df["week_sin26"] = np.sin(4.0 * np.pi * woy / 52.0)
    df["week_cos26"] = np.cos(4.0 * np.pi * woy / 52.0)
    df["time_idx"] = df["week_idx"].astype(float)
    df["atc_share_lag1"] = df.groupby("atc_code")["atc_share_raw"].shift(1)
    df["is_holiday_zero_lag1"] = df.groupby("atc_code")["is_holiday_zero_obs"].shift(1)
    f2_cols = [
        "week_sin52", "week_cos52", "week_sin26", "week_cos26",
        "time_idx", "is_rs_holiday", "atc_share_lag1", "is_holiday_zero_lag1",
    ]

    # F3 — ERA5 Weather (11 đồng kỳ + 8 trễ = 19 tổng; Compact = 7 biến trực giao vật lý)
    df["temp_range"] = df["temp_max"] - df["temp_min"]
    df["hdd"] = np.maximum(0.0, 18.0 - df["temp_mean"])
    df["cdd"] = np.maximum(0.0, df["temp_mean"] - 22.0)
    df["temp_shock_2w"] = df["temp_mean"] - df.groupby("atc_code")["temp_mean"].shift(2)

    df["temp_mean_lag1"] = df.groupby("atc_code")["temp_mean"].shift(1)
    df["temp_mean_lag2"] = df.groupby("atc_code")["temp_mean"].shift(2)
    df["temp_min_lag1"] = df.groupby("atc_code")["temp_min"].shift(1)
    df["hdd_lag1"] = df.groupby("atc_code")["hdd"].shift(1)
    df["hdd_lag2"] = df.groupby("atc_code")["hdd"].shift(2)
    df["humidity_mean_lag1"] = df.groupby("atc_code")["humidity_mean"].shift(1)
    df["precip_sum_lag1"] = df.groupby("atc_code")["precip_sum"].shift(1)
    df["daylight_duration_s_lag1"] = df.groupby("atc_code")["daylight_duration_s"].shift(1)

    f3_compact_cols = [
        "temp_mean", "hdd", "humidity_mean", "daylight_duration_s",
        "temp_shock_2w", "temp_mean_lag1", "humidity_mean_lag1",
    ]
    f3_lagged_only_cols = [
        "temp_mean_lag1", "temp_mean_lag2", "temp_min_lag1", "hdd_lag1", "hdd_lag2",
        "humidity_mean_lag1", "precip_sum_lag1", "daylight_duration_s_lag1",
    ]
    f3_all_cols = [
        "temp_mean", "temp_min", "temp_max", "humidity_mean", "precip_sum",
        "pressure_msl_mean", "daylight_duration_s", "temp_range", "hdd", "cdd", "temp_shock_2w",
    ] + f3_lagged_only_cols

    return df.drop(columns=["atc_share_raw"]), f1_cols, f2_cols, f3_compact_cols, f3_lagged_only_cols, f3_all_cols


def build_daily_features(df_in: pd.DataFrame):
    df = df_in.sort_values(["atc_code", "local_date"]).copy()
    d_f1 = []
    for k in [1, 2, 3, 7, 14, 28, 364]:
        c = f"lag_{k}"
        df[c] = df.groupby("atc_code")["qty"].shift(k)
        d_f1.append(c)

    s1 = df.groupby("atc_code")["qty"].shift(1)
    df["roll_mean_7"] = s1.groupby(df["atc_code"]).transform(lambda s: s.rolling(7, min_periods=7).mean())
    df["roll_std_7"] = s1.groupby(df["atc_code"]).transform(lambda s: s.rolling(7, min_periods=7).std())
    df["roll_mean_28"] = s1.groupby(df["atc_code"]).transform(lambda s: s.rolling(28, min_periods=28).mean())
    df["ewma_7"] = s1.groupby(df["atc_code"]).transform(lambda s: s.ewm(span=7, adjust=False, min_periods=1).mean())
    d_f1 += ["roll_mean_7", "roll_std_7", "roll_mean_28", "ewma_7"]

    dow = df["local_date"].dt.dayofweek.astype(float)
    doy = df["local_date"].dt.dayofyear.astype(float)
    df["dow_sin"] = np.sin(2.0 * np.pi * dow / 7.0)
    df["dow_cos"] = np.cos(2.0 * np.pi * dow / 7.0)
    df["doy_sin"] = np.sin(2.0 * np.pi * doy / 365.25)
    df["doy_cos"] = np.cos(2.0 * np.pi * doy / 365.25)
    df["time_idx"] = df["day_idx"].astype(float)
    df["is_holiday_zero_lag1"] = df.groupby("atc_code")["is_holiday_zero_obs"].shift(1)
    d_f2 = ["dow_sin", "dow_cos", "doy_sin", "doy_cos", "time_idx", "is_rs_holiday", "is_holiday_zero_lag1"]

    df["temp_range"] = df["temp_max"] - df["temp_min"]
    df["hdd"] = np.maximum(0.0, 18.0 - df["temp_mean"])
    df["cdd"] = np.maximum(0.0, df["temp_mean"] - 22.0)
    df["temp_mean_lag7"] = df.groupby("atc_code")["temp_mean"].shift(7)
    df["hdd_lag7"] = df.groupby("atc_code")["hdd"].shift(7)
    df["humidity_mean_lag7"] = df.groupby("atc_code")["humidity_mean"].shift(7)
    d_f3 = [
        "temp_mean", "temp_min", "temp_max", "humidity_mean", "precip_sum",
        "pressure_msl_mean", "daylight_duration_s", "temp_range", "hdd", "cdd",
        "temp_mean_lag7", "hdd_lag7", "humidity_mean_lag7",
    ]
    return df, d_f1, d_f2, d_f3


df_feat, f1_cols, f2_cols, f3_compact_cols, f3_lagged_only_cols, f3_all_cols = build_weekly_features(df_weekly)
df_daily_feat, d_f1_cols, d_f2_cols, d_f3_cols = build_daily_features(df_daily_base)

weekly_all_cols = f1_cols + f2_cols + f3_all_cols
weekly_compact_cols = f1_cols + f2_cols + f3_compact_cols
daily_all_cols = d_f1_cols + d_f2_cols + d_f3_cols

forbidden = {"qty", "qty_rate", "active_days", "is_holiday_zero", "is_holiday_zero_obs"}
assert not (set(weekly_all_cols) & forbidden)
assert not (set(daily_all_cols) & forbidden)
print(f"Weekly features -> F1={len(f1_cols)}, F2={len(f2_cols)}, F3_compact={len(f3_compact_cols)}, F3_lagOnly={len(f3_lagged_only_cols)}, F3_all={len(f3_all_cols)}")
print(f"Daily features  -> F1={len(d_f1_cols)}, F2={len(d_f2_cols)}, F3={len(d_f3_cols)}")

In [ ]:
# 4. Automated Perturbation Leakage Unit-Tests (Weekly & Daily) + Diagnostics

df_w_mut = df_weekly.copy()
df_w_mut.loc[df_w_mut["week_idx"] >= 250, "qty"] = df_w_mut.loc[df_w_mut["week_idx"] >= 250, "qty"] * 99.0 + 123.4
df_w_feat_mut, _, _, _, _, _ = build_weekly_features(df_w_mut)

m_w = df_feat["week_idx"] <= 250
diff_w = float((df_feat.loc[m_w, weekly_all_cols].fillna(-9999.0) - df_w_feat_mut.loc[m_w, weekly_all_cols].fillna(-9999.0)).abs().max().max())
assert diff_w == 0.0, f"WEEKLY TARGET LEAKAGE DETECTED! max_diff={diff_w}"
assert df_feat.loc[df_feat["week_idx"] >= 52, weekly_all_cols].isna().sum().sum() == 0, "Weekly NaNs after 52W warmup!"

df_d_mut = df_daily_base.copy()
df_d_mut.loc[df_d_mut["day_idx"] >= 1741, "qty"] = df_d_mut.loc[df_d_mut["day_idx"] >= 1741, "qty"] * 99.0 + 45.6
df_d_feat_mut, _, _, _ = build_daily_features(df_d_mut)

m_d = df_daily_feat["day_idx"] <= 1741
diff_d = float((df_daily_feat.loc[m_d, daily_all_cols].fillna(-9999.0) - df_d_feat_mut.loc[m_d, daily_all_cols].fillna(-9999.0)).abs().max().max())
assert diff_d == 0.0, f"DAILY TARGET LEAKAGE DETECTED! max_diff={diff_d}"
assert df_daily_feat.loc[df_daily_feat["day_idx"] >= 364, daily_all_cols].isna().sum().sum() == 0, "Daily NaNs after 364D warmup!"

print(f"PERTURBATION LEAKAGE UNIT-TEST PASS -> Weekly max_abs_diff={diff_w}, Daily max_abs_diff={diff_d}")

# VIF check strictly on Train+Val (W53..W250) comparing compact ERA5 features
tv_mask = (df_feat["week_idx"] >= 52) & (df_feat["week_idx"] < 250)
X_tv = df_feat.loc[tv_mask, weekly_compact_cols].dropna()
corr_target = X_tv.apply(lambda col: df_feat.loc[tv_mask, "qty_rate"].corr(col, method="spearman")).sort_values(key=abs, ascending=False)
print("\nTop 10 Weekly features by |Spearman corr| with qty_rate on Train+Val (W53..W250):")
print(corr_target.head(10).round(3).to_string())

vif_cols = ["lag_1", "lag_4", "lag_52", "rolling_mean_4", "week_sin52", "week_cos52", "temp_mean", "hdd", "humidity_mean", "daylight_duration_s", "atc_share_lag1"]
X_vif = np.column_stack([np.ones(len(X_tv)), X_tv[vif_cols].to_numpy(dtype=float)])
vif_df = pd.DataFrame([{"feature": c, "VIF": round(float(variance_inflation_factor(X_vif, i + 1)), 2)} for i, c in enumerate(vif_cols)])
print("\nCompact feature VIF on Train+Val:")
print(vif_df.to_string(index=False))

p_fs_w = ART_DIR / "feature_store_weekly.parquet"
p_fs_d = ART_DIR / "feature_store_daily.parquet"
df_feat.to_parquet(p_fs_w, index=False)
df_daily_feat.to_parquet(p_fs_d, index=False)

catalog = {
    "pipeline": "Unified 5-Notebook PharmaFlow Forecasting Flow",
    "exposure_col": "active_days",
    "target_col": "qty",
    "target_rate_col": "qty_rate",
    "f1_cols": f1_cols,
    "f2_cols": f2_cols,
    "f3_compact_cols": f3_compact_cols,
    "f3_lagged_only_cols": f3_lagged_only_cols,
    "f3_all_cols": f3_all_cols,
    "weekly_compact_feature_cols": weekly_compact_cols,
    "weekly_all_feature_cols": weekly_all_cols,
    "daily_f1_f2_cols": d_f1_cols + d_f2_cols,
    "daily_f3_cols": d_f3_cols,
    "daily_all_feature_cols": daily_all_cols,
    "warmup_weeks": 52,
    "warmup_days": 364,
    "leakage_tests": {
        "is_holiday_zero_removed": True,
        "bfill_removed": True,
        "active_days_not_in_tree_features": True,
        "weekly_perturbation_diff": diff_w,
        "daily_perturbation_diff": diff_d,
    },
}
(ART_DIR / "feature_catalog.json").write_text(json.dumps(catalog, indent=2), encoding="utf-8")
print(f"\nSaved Weekly {df_feat.shape}, Daily {df_daily_feat.shape}, and feature_catalog.json.")